In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 36.0 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
# pandas print option
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

In [ ]:
# Data
df = pd.read_csv('data/fluorclass_dataset.csv',encoding='utf-8-sig')
display(df)

,SMILES,Class
0,Nc1c(/N=N/c2ccc([N+](=O)[O-])cc2)c(S(=O)(=O)[O][Na])cc2c1C(=O)/C(=N\Nc1ccccc1)C(S(=O)(=O)[O][Na])=C2,0
1,Nc1c(/N=N/c2ccc(S(=O)(=O)[O][Na])cc2)cc(S(=O)(=O)[O][Na])c2c1C(=O)/C(=N\Nc1ccc([N+](=O)[O-])cc1)C=C2,0
2,Nc1c(/N=N/c2ccc(S(=O)(=O)[O][Na])cc2)cc(S(=O)(=O)[O][Na])c2c1C(=O)/C(=N\Nc1cccc3ccccc13)C=C2,0
3,O=[N+]([O-])c1ccc2c(/N=N/c3c(O)ccc4ccccc34)c(O)cc(S(=O)(=O)[O][Na])c2c1,0
4,Nc1ccc(S(=O)(=O)[O][Na])c2c1C(=O)/C(=N\Nc1ccc(/N=N/c3ccc(S(=O)(=O)[O][Na])cc3)c3ccccc13)C(S(=O)(=O)[O][Na])=C2,0
...,...,...
5461,CN(C)c1ccc(-c2ccc3c(=O)c(O)coc3c2)cc1,1
5462,COc1coc2cc(-c3ccc(N(C)C)cc3)ccc2c1=O,1
5463,COC(=O)CN1C(=O)c2cc3ccc(N(C)C)cc3cc2C1=O,1
5464,CCN(CC)c1ccc2c(c1)C(C)(C)c1cc(C=O)ccc1-2,1


In [ ]:
smiles_list = df['SMILES'].tolist()
labels = df['Class'].values
indices = df.index.tolist()

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

# Morgan Fingerprint (ECFP) based Feature Extraction ---------------------------

# SMILES to ECFP Function
def smiles_to_fingerprint(smiles, radius=4, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    if mol:
        return list(fpgen.GetFingerprint(mol))
    else:
        return [0] * n_bits  # return to zero vector on conversion failure

# Convert SMILES to ECFP and Numpy array
X = np.array([smiles_to_fingerprint(smiles) for smiles in smiles_list])
y = np.array(labels)

# Check
print(f"\nMorgan Fingerprints Feature Shape (2048 bits): {X.shape}")
print(f"Labels Shape: {y.shape}")
df_X = pd.DataFrame(X, columns=[f'bit_{i}' for i in range(X.shape[1])])
display(df_X)


Morgan Fingerprints Feature Shape (2048 bits): (5466, 2048)
Labels Shape: (5466,)


,bit_0,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2038,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5461,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5462,0,0,0,0,0,0,0,0,1,0,...,0,0,1,1,0,0,0,0,0,0
5463,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5464,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
import time
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# Random Split Repeated Learning ===============================================
results = []

for run in range(1, 101):

    print(f"\n========== Run {run}/100 ==========")

    # Random seed
    seed = run
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Train-Test Split
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X,
        y,
        indices,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    # Model Initialization
    tf.keras.backend.clear_session()

    input_layer = Input(shape=(2048,), name='input')

    x = Dense(128, activation='relu', name='dense_1')(input_layer)
    x = Dropout(0.3, name='dropout_1')(x)

    x = Dense(64, activation='relu', name='dense_2')(x)
    x = Dropout(0.3, name='dropout_2')(x)

    output_layer = Dense(1,activation='sigmoid',name='dense_3')(x)

    model = Model(inputs=input_layer,outputs=output_layer)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['binary_accuracy']
    )

    # Callback
    earlystopping = EarlyStopping(
        monitor='val_loss',
        patience=200,
        mode='min',
        restore_best_weights=True,
        verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=100,
        min_lr=1e-7,
        verbose=0
    )

    # Train Learning
    start_time = time.time()

    hist = model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=20000,
        batch_size=64,
        callbacks=[earlystopping, reduce_lr],
        verbose=0
    )

    training_time = time.time() - start_time

    # Test prediction
    y_pred_test = model.predict(X_test,verbose=0).flatten()
    y_pred_labels_test = (y_pred_test >= 0.5).astype(int)

    # Performance Evaluation
    accuracy = accuracy_score(y_test,y_pred_labels_test)
    precision = precision_score(y_test, y_pred_labels_test)
    recall = recall_score(y_test, y_pred_labels_test)
    f1 = f1_score(y_test, y_pred_labels_test)
    auc = roc_auc_score(y_test, y_pred_test)


    # Save the Result
    results.append({
        'Run': run,
        'Seed': seed,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'AUC': auc,
        'Epochs': len(hist.history['loss']),
        'Training_time_sec': training_time
    })

    print(
        f"Accuracy = {accuracy:.4f} | "
        f"Precision = {precision:.4f} | "
        f"Recall = {recall:.4f} | "
        f"F1 = {f1:.4f} | "
        f"AUC = {auc:.4f} | "
        f"Epochs = {len(hist.history['loss'])}"
    )

    # 메모리 정리
    del model
    gc.collect()

# Result DataFrame
results_df = pd.DataFrame(results)
display(results_df)


========== Run 1/100 ==========
Accuracy = 0.9845 | Precision = 0.9813 | Recall = 0.9897 | F1 = 0.9855 | AUC = 0.9973 | Epochs = 221

========== Run 2/100 ==========
Accuracy = 0.9817 | Precision = 0.9763 | Recall = 0.9897 | F1 = 0.9830 | AUC = 0.9982 | Epochs = 231

========== Run 3/100 ==========
Accuracy = 0.9835 | Precision = 0.9732 | Recall = 0.9966 | F1 = 0.9847 | AUC = 0.9979 | Epochs = 215

========== Run 4/100 ==========
Accuracy = 0.9890 | Precision = 0.9864 | Recall = 0.9931 | F1 = 0.9897 | AUC = 0.9994 | Epochs = 221

========== Run 5/100 ==========
Accuracy = 0.9845 | Precision = 0.9829 | Recall = 0.9880 | F1 = 0.9855 | AUC = 0.9987 | Epochs = 214

========== Run 6/100 ==========
Accuracy = 0.9781 | Precision = 0.9778 | Recall = 0.9811 | F1 = 0.9795 | AUC = 0.9976 | Epochs = 215

========== Run 7/100 ==========
Accuracy = 0.9845 | Precision = 0.9829 | Recall = 0.9880 | F1 = 0.9855 | AUC = 0.9991 | Epochs = 228

========== Run 8/100 ==========
Accuracy = 0.9826 | Precision

,Run,Seed,Accuracy,Precision,Recall,F1,AUC,Epochs,Training_time_sec
0,1,1,0.984461,0.981293,0.989708,0.985482,0.997340,221,114.628618
1,2,2,0.981718,0.976311,0.989708,0.982964,0.998187,231,108.630744
2,3,3,0.983547,0.973199,0.996569,0.984746,0.997912,215,102.488313
3,4,4,0.989031,0.986371,0.993139,0.989744,0.999403,221,101.355059
4,5,5,0.984461,0.982935,0.987993,0.985458,0.998711,214,97.717069
...,...,...,...,...,...,...,...,...,...
95,96,96,0.981718,0.974705,0.991424,0.982993,0.997872,216,96.785247
96,97,97,0.983547,0.978003,0.991424,0.984668,0.997394,240,105.925394
97,98,98,0.980804,0.977891,0.986278,0.982067,0.998261,238,108.574693
98,99,99,0.986289,0.982993,0.991424,0.987190,0.998785,219,91.041579


In [ ]:
# Result DataFrame
results_df = pd.DataFrame(results)
display(results_df)

,Run,Seed,Accuracy,Precision,Recall,F1,AUC,Epochs,Training_time_sec
0,1,1,0.984461,0.981293,0.989708,0.985482,0.997340,221,114.628618
1,2,2,0.981718,0.976311,0.989708,0.982964,0.998187,231,108.630744
2,3,3,0.983547,0.973199,0.996569,0.984746,0.997912,215,102.488313
3,4,4,0.989031,0.986371,0.993139,0.989744,0.999403,221,101.355059
4,5,5,0.984461,0.982935,0.987993,0.985458,0.998711,214,97.717069
...,...,...,...,...,...,...,...,...,...
95,96,96,0.981718,0.974705,0.991424,0.982993,0.997872,216,96.785247
96,97,97,0.983547,0.978003,0.991424,0.984668,0.997394,240,105.925394
97,98,98,0.980804,0.977891,0.986278,0.982067,0.998261,238,108.574693
98,99,99,0.986289,0.982993,0.991424,0.987190,0.998785,219,91.041579


In [ ]:
# Save
results_df.to_csv('result/03_Classification/repeated_evaluation_result.csv', index=False)

In [ ]:
# Summary of Repeated Result
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

summary = pd.DataFrame({
    'Mean': results_df[metrics].mean(),
    'SD': results_df[metrics].std(),
    'Min': results_df[metrics].min(),
    'Max': results_df[metrics].max()
})

print(summary)

print("\n===== Mean ± SD =====")

for metric in metrics:
    mean = results_df[metric].mean()
    sd = results_df[metric].std()

    print(f"{metric}: {mean:.4f} ± {sd:.4f}")

               Mean        SD       Min       Max
Accuracy   0.983318  0.003773  0.975320  0.992687
Precision  0.978269  0.005769  0.965000  0.993139
Recall     0.990738  0.003422  0.981132  0.998285
F1         0.984453  0.003493  0.977021  0.993162
AUC        0.998356  0.000735  0.995353  0.999785

===== Mean ± SD =====
Accuracy: 0.9833 ± 0.0038
Precision: 0.9783 ± 0.0058
Recall: 0.9907 ± 0.0034
F1: 0.9845 ± 0.0035
AUC: 0.9984 ± 0.0007


In [ ]:
summary.to_csv("result/03_Classification/repeated_evaluation_summary.csv",index=True,encoding="utf-8-sig")